In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "tatarstan_economic_data_clean.csv"
)

df = pd.read_csv(DATA_PATH)
df.head()

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


,year,unemployment_rate,investment_index,average_salary,housing_price,grp_mln_rub,working_age_population_thousands,unemployment_diff,investment_diff,grp_log_growth,salary_log_growth,housing_price_log_growth
0,2000,12.1,124.8,2180.0,2850.0,168590.0,2655.0,NaN,NaN,NaN,NaN,NaN
1,2001,10.8,118.6,1850.0,3210.0,194230.0,2662.0,-1.3,-6.2,0.141573,-0.164139,0.118952
2,2002,9.6,115.4,3560.0,3680.0,223450.0,2670.0,-1.2,-3.2,0.140145,0.654575,0.136642
3,2003,8.4,120.2,4420.0,4150.0,261870.0,2675.0,-1.2,4.8,0.158661,0.216379,0.120196
4,2004,7.5,126.7,5580.0,4890.0,312450.0,2680.0,-0.9,6.5,0.176596,0.233049,0.164084


In [2]:
model_columns = [
    "year",
    "unemployment_diff",
    "investment_diff",
    "grp_log_growth"
]
model_df = (
    df[model_columns]
    .dropna()
    .reset_index(drop=True)
)

print("Model observations:", len(model_df))
model_df.head()

Model observations: 25


,year,unemployment_diff,investment_diff,grp_log_growth
0,2001,-1.3,-6.2,0.141573
1,2002,-1.2,-3.2,0.140145
2,2003,-1.2,4.8,0.158661
3,2004,-0.9,6.5,0.176596
4,2005,-0.6,3.6,0.192881


In [3]:
y = model_df["unemployment_diff"].values
X = model_df[["investment_diff", "grp_log_growth"]].copy()

X_mean = X.mean()
X_std = X.std()
X_scaled = (X - X_mean) / X_std

print("Target:")
print("unemployment_diff")
print("\nFeatures:")
print(X.columns.tolist())
print("\nStandardized features:")

X_scaled.head()

Target:
unemployment_diff

Features:
['investment_diff', 'grp_log_growth']

Standardized features:


,investment_diff,grp_log_growth
0,-0.470284,0.545408
1,-0.210027,0.524863
2,0.483991,0.791146
3,0.631470,1.049086
4,0.379888,1.283278


In [4]:
with pm.Model() as model:
    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=1
    )
    beta_investment = pm.Normal(
        "beta_investment",
        mu=0,
        sigma=1
    )
    beta_grp = pm.Normal(
        "beta_grp",
        mu=0,
        sigma=1
    )
    sigma = pm.HalfNormal(
        "sigma",
        sigma=1
    )
    mu = (
        intercept
        + beta_investment * X_scaled["investment_diff"].values
        + beta_grp * X_scaled["grp_log_growth"].values
    )
    unemployment_obs = pm.Normal(
        "unemployment_obs",
        mu=mu,
        sigma=sigma,
        observed=y
    )

    trace = pm.sample(
        draws=2000,
        tune=1000,
        chains=2,
        target_accept=0.95,
        random_seed=42
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, beta_investment, beta_grp, sigma]


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 23 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [5]:
summary = az.summary(
    trace,
    var_names=[
        "intercept",
        "beta_investment",
        "beta_grp",
        "sigma"
    ],
    hdi_prob=0.94
)
summary

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
intercept,-0.411,0.082,-0.563,-0.259,0.001,0.002,4383.0,2792.0,1.0
beta_investment,-0.216,0.093,-0.397,-0.045,0.002,0.002,3376.0,2617.0,1.0
beta_grp,-0.411,0.094,-0.582,-0.232,0.002,0.002,3295.0,2439.0,1.0
sigma,0.406,0.067,0.291,0.528,0.001,0.001,3384.0,2461.0,1.0


A Bayesian linear regression was fitted to explain annual changes in the unemployment rate using:
- annual change in the investment index;
- GRP log growth
Both predictors show a negative association with changes in unemployment.
The posterior mean for the investment coefficient is -0.216, while the posterior mean for GRP growth is -0.411.
The 94% credible intervals for both coefficients are below zero.
MCMC diagnostics indicate good convergence, with R-hat values of 1.00 and high effective sample sizes.

Для объяснения ежегодных изменений уровня безработицы была использована байесовская линейная регрессия с использованием:
- годового изменения индекса инвестиций;
- логарифмического роста валового продукта.
Оба предиктора демонстрируют отрицательную корреляцию с изменениями уровня безработицы.
Среднее апостериорное значение коэффициента для инвестиций составляет -0,216, а среднее апостериорное значение коэффициента для роста валового продукта — -0,411.
94%-ные доверительные интервалы для обоих коэффициентов находятся ниже нуля.
Диагностика методом Монте-Карло с цепью Маркова показывает хорошую сходимость: значения R-hat равны 1,00, а эффективный размер выборки высок.